1. Import librairies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pathlib import Path


2.Import documents

In [2]:
#name = pd.read_csv(r"imdb\name.basics.tsv",  sep="\t",na_values='\\N', usecols=['nconst', 'primaryName', 'knownForTitles']) 
name = pd.read_csv(r"..\..\data\raw\imdb\name.basics.tsv", sep="\t", na_values='\\N', usecols=['nconst', 'primaryName', 'knownForTitles'])


In [ ]:
display(name.describe())

In [3]:
title = pd.read_csv(r'..\..\data\raw\imdb\title.basics.tsv',sep='\t',na_values='\\N',usecols=['tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'startYear', 'genres'],on_bad_lines='skip',engine='python')
title = title[title['titleType'] == 'movie']

In [4]:
Principal = pd.read_csv(r'..\..\data\raw\imdb\title.principals.tsv', sep='\t', na_values='\\N', usecols=['tconst', 'nconst', 'category'])

In [5]:
Ratings = pd.read_csv(r'..\..\data\raw\imdb\title.ratings.tsv', sep='\t',na_values='\\N')

3 Merge df_films et df_cast 

In [6]:
df_films = title.merge(Ratings, on='tconst',how='left')
df_cast = Principal.merge(name, on='nconst', how='left')
df_cast = df_cast[df_cast['tconst'].isin(df_films['tconst'])]

In [7]:
print(f'Films shape {df_films.shape}')
print(f'Casting shape:{df_cast.shape}')



Films shape (742654, 8)
Casting shape:(8551946, 5)


In [ ]:
df_films.head(10)


In [8]:
df_films.isna().sum()
print(f"Nan runtimeMinutes: {df_films['runtimeMinutes'].isna().mean()*100:.1f}%de NaN")
print(f"Nan averageRatings: {df_films['averageRating'].isna().mean()*100:.1f}%de NaN")
print(f"Nan numVotes: {df_films['numVotes'].isna().mean()*100:.1f}%de NaN")
print(f"Nan genres: {df_films['genres'].isna().mean()*100:.1f}%de NaN")
print(f"Nan startYear: {df_films['startYear'].isna().mean()*100:.1f}%de NaN")

print(f"Total ligne: {len(df_films)}")

Nan runtimeMinutes: 36.9%de NaN
Nan averageRatings: 54.0%de NaN
Nan numVotes: 54.0%de NaN
Nan genres: 10.5%de NaN
Nan startYear: 15.0%de NaN
Total ligne: 742654


54% de NA sur averageRating ainsi que numVotes, aucun interet de garder des films sans votes
Meme chose pour les Nan genre , 10.5% que nous supprimons 

In [10]:
df_films = df_films.dropna(subset='averageRating')
df_films = df_films.dropna(subset='numVotes')
df_films = df_films.dropna(subset='genres')

df_films.shape

(330702, 8)

Nous passons de 742654 films a 341961 films après suppression de NA sur la colonne averageRating et numVotes

In [ ]:
sns.histplot(data=df_films['averageRating'], kde=True)
#Nous avons une note moyenne de 6.5

In [ ]:
df_films[df_films['averageRating'].isna()].groupby('startYear').size().plot()

In [11]:
df_films.to_csv('df_films_clean.csv', index=False)